In [6]:
from helpers import init, preprocess_data

In [7]:
SEED = 42
X_train_raw, y_train_cm, X_val_raw, y_val_cm = init(SEED)
X_train, y_train, X_val, y_val = preprocess_data(X_train_raw, y_train_cm, X_val_raw, y_val_cm)

PyTorch: 2.11.0+cu130
Training rows: 42166
Validation rows: 4685
Input shape: (42166, 9)
Target x range (cm): 0.0 to 281.0
Target y range (cm): 0.0 to 275.0
Using split files:
  train: train_clean_3x3_1cm.csv 42166 rows
  validation: validation_clean_3x3_1cm.csv 4685 rows
RSS scale: 0.8493868112564087
Target min cm: [0. 0.]
Target range cm: [281. 275.]


# Get the model

In [8]:
from vlp_hackathon.baseline_model import BaselineMLP
model_cls = BaselineMLP

# Run a trail

In [9]:
from torch import nn
import torch
import numpy as np
from matplotlib import pyplot as plt

def train_model(model, optimizer, loss_fn, epochs, batch_size, plot: bool = False):
    X_train_t = torch.from_numpy(X_train)
    y_train_t = torch.from_numpy(y_train)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)

    history = []
    train_rng = np.random.default_rng(SEED)
    for epoch in range(epochs):
        model.train()
        permutation = train_rng.permutation(len(X_train))
        running = 0.0
        for start in range(0, len(permutation), batch_size):
            idx = permutation[start:start + batch_size]
            xb = X_train_t[idx]
            yb = y_train_t[idx]
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += float(loss.item()) * len(idx)

        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(X_val_t), y_val_t).item())
        history.append((running / len(X_train), val_loss))
        print(f"epoch={epoch + 1:02d} train_mse={history[-1][0]:.6f} val_mse={val_loss:.6f}")

    history = np.asarray(history, dtype=np.float32)
    if plot:
        plt.figure(figsize=(6, 4))
        plt.plot(history[:, 0], label="train")
        plt.plot(history[:, 1], label="validation")
        plt.xlabel("Epoch")
        plt.ylabel("MSE on normalized coordinates")
        plt.legend()
        plt.title("Baseline training curve")
        plt.show()

    return np.min(history[:, 1])

def run_trial(trial):
    epochs = 50
    batch_size = 1024

    print(trial)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    model = BaselineMLP(input_features=X_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    return train_model(model, optimizer, loss_fn, epochs, batch_size)



In [12]:
import optuna
from optuna.trial import TrialState

study = optuna.create_study(direction='maximize')
study.optimize(run_trial, n_trials=10)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2026-07-08 14:22:13,169] A new study created in memory with name: no-name-a0ec8be8-b415-4c3f-9bd9-a312eaf5e003


epoch=01 train_mse=0.089511 val_mse=0.073470
epoch=02 train_mse=0.047634 val_mse=0.019075
epoch=03 train_mse=0.011539 val_mse=0.007478
epoch=04 train_mse=0.007302 val_mse=0.006258
epoch=05 train_mse=0.006217 val_mse=0.005307
epoch=06 train_mse=0.005215 val_mse=0.004387
epoch=07 train_mse=0.004296 val_mse=0.003624
epoch=08 train_mse=0.003600 val_mse=0.003111
epoch=09 train_mse=0.003122 val_mse=0.002745
epoch=10 train_mse=0.002755 val_mse=0.002446
epoch=11 train_mse=0.002453 val_mse=0.002192
epoch=12 train_mse=0.002200 val_mse=0.001984
epoch=13 train_mse=0.001987 val_mse=0.001805
epoch=14 train_mse=0.001791 val_mse=0.001618
epoch=15 train_mse=0.001602 val_mse=0.001447
epoch=16 train_mse=0.001429 val_mse=0.001292
epoch=17 train_mse=0.001269 val_mse=0.001146
epoch=18 train_mse=0.001126 val_mse=0.001020
epoch=19 train_mse=0.001001 val_mse=0.000911
epoch=20 train_mse=0.000897 val_mse=0.000818
epoch=21 train_mse=0.000814 val_mse=0.000749
epoch=22 train_mse=0.000748 val_mse=0.000690
epoch=23 t

[I 2026-07-08 14:22:14,658] Trial 0 finished with value: 0.00024539465084671974 and parameters: {'lr': 0.0015711693869537393}. Best is trial 0 with value: 0.00024539465084671974.


epoch=50 train_mse=0.000262 val_mse=0.000245
epoch=01 train_mse=0.096228 val_mse=0.094943
epoch=02 train_mse=0.095310 val_mse=0.093908
epoch=03 train_mse=0.094167 val_mse=0.092573
epoch=04 train_mse=0.092721 val_mse=0.090892
epoch=05 train_mse=0.090824 val_mse=0.088639
epoch=06 train_mse=0.088232 val_mse=0.085588
epoch=07 train_mse=0.084836 val_mse=0.081742
epoch=08 train_mse=0.080665 val_mse=0.077205
epoch=09 train_mse=0.075880 val_mse=0.072096
epoch=10 train_mse=0.070534 val_mse=0.066458
epoch=11 train_mse=0.064612 val_mse=0.060189
epoch=12 train_mse=0.058259 val_mse=0.053827
epoch=13 train_mse=0.052016 val_mse=0.047740
epoch=14 train_mse=0.046024 val_mse=0.041800
epoch=15 train_mse=0.040199 val_mse=0.036219
epoch=16 train_mse=0.034872 val_mse=0.031222
epoch=17 train_mse=0.030148 val_mse=0.026842
epoch=18 train_mse=0.026033 val_mse=0.023084
epoch=19 train_mse=0.022527 val_mse=0.019924
epoch=20 train_mse=0.019605 val_mse=0.017323
epoch=21 train_mse=0.017222 val_mse=0.015235
epoch=22 t

[I 2026-07-08 14:22:16,145] Trial 1 finished with value: 0.005806653294712305 and parameters: {'lr': 0.00013814968138023615}. Best is trial 1 with value: 0.005806653294712305.


...

epoch=50 train_mse=0.000058 val_mse=0.000038
epoch=01 train_mse=0.016671 val_mse=0.003060
epoch=02 train_mse=0.001938 val_mse=0.000986
epoch=03 train_mse=0.000735 val_mse=0.000559
epoch=04 train_mse=0.000506 val_mse=0.000415
epoch=05 train_mse=0.000394 val_mse=0.000389
epoch=06 train_mse=0.000360 val_mse=0.000290
epoch=07 train_mse=0.000309 val_mse=0.000254
epoch=08 train_mse=0.000247 val_mse=0.000221
epoch=09 train_mse=0.000248 val_mse=0.000212
epoch=10 train_mse=0.000215 val_mse=0.000190
epoch=11 train_mse=0.000180 val_mse=0.000154
epoch=12 train_mse=0.000212 val_mse=0.000246
epoch=13 train_mse=0.000165 val_mse=0.000165
epoch=14 train_mse=0.000180 val_mse=0.000162
epoch=15 train_mse=0.000148 val_mse=0.000144
epoch=16 train_mse=0.000158 val_mse=0.000162
epoch=17 train_mse=0.000170 val_mse=0.000127
epoch=18 train_mse=0.000114 val_mse=0.000105
epoch=19 train_mse=0.000159 val_mse=0.000122
epoch=20 train_mse=0.000138 val_mse=0.000133
epoch=21 train_mse=0.000109 val_mse=0.000111
epoch=22 t

[I 2026-07-08 14:22:26,792] Trial 8 finished with value: 5.7465564168524e-05 and parameters: {'lr': 0.05173289130961395}. Best is trial 5 with value: 0.08369835466146469.


epoch=46 train_mse=0.000109 val_mse=0.000093
epoch=47 train_mse=0.000089 val_mse=0.000086
epoch=48 train_mse=0.000089 val_mse=0.000057
epoch=49 train_mse=0.000081 val_mse=0.000095
epoch=50 train_mse=0.000072 val_mse=0.000069
epoch=01 train_mse=0.016357 val_mse=0.002290
epoch=02 train_mse=0.001342 val_mse=0.000861
epoch=03 train_mse=0.000622 val_mse=0.000486
epoch=04 train_mse=0.000455 val_mse=0.000367
epoch=05 train_mse=0.000344 val_mse=0.000295
epoch=06 train_mse=0.000311 val_mse=0.000229
epoch=07 train_mse=0.000219 val_mse=0.000194
epoch=08 train_mse=0.000197 val_mse=0.000231
epoch=09 train_mse=0.000193 val_mse=0.000209
epoch=10 train_mse=0.000230 val_mse=0.000144
epoch=11 train_mse=0.000146 val_mse=0.000175
epoch=12 train_mse=0.000154 val_mse=0.000158
epoch=13 train_mse=0.000134 val_mse=0.000148
epoch=14 train_mse=0.000163 val_mse=0.000144
epoch=15 train_mse=0.000142 val_mse=0.000202
epoch=16 train_mse=0.000122 val_mse=0.000111
epoch=17 train_mse=0.000109 val_mse=0.000145
epoch=18 t

[I 2026-07-08 14:22:28,368] Trial 9 finished with value: 5.320608397596516e-05 and parameters: {'lr': 0.050152095490538125}. Best is trial 5 with value: 0.08369835466146469.


epoch=45 train_mse=0.000072 val_mse=0.000064
epoch=46 train_mse=0.000089 val_mse=0.000088
epoch=47 train_mse=0.000086 val_mse=0.000058
epoch=48 train_mse=0.000062 val_mse=0.000063
epoch=49 train_mse=0.000069 val_mse=0.000075
epoch=50 train_mse=0.000072 val_mse=0.000054
Study statistics: 
  Number of finished trials:  10
  Number of pruned trials:  0
  Number of complete trials:  10
Best trial:
  Value:  0.08369835466146469
  Params: 
    lr: 1.7987092334414985e-05
